In [17]:
import numpy as np
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder



EFFECTIVENESS = 1.0   
REVIEW_COST = 5.0     

In [9]:
def cost_opt_threshold(y, s, amt):
    o = np.argsort(-s, kind="stable")
    ss, ys, am = s[o], y[o], amt[o]
    total = float((y * amt).sum())
    cost = total - EFFECTIVENESS * np.cumsum(ys * am) + REVIEW_COST * np.arange(1, len(s) + 1)
    last = np.flatnonzero(np.r_[ss[1:] != ss[:-1], True])
    j = last[np.argmin(cost[last])]
    return np.inf if cost[j] >= total else ss[j]

y   = np.array([1,    0,    1,    0,    0])
s   = np.array([0.95, 0.80, 0.60, 0.40, 0.10])
amt = np.array([500,  50,   200,  1000, 20])

opt_threshold = cost_opt_threshold(y, s, amt)

print(f"Tổng gian lận ban đầu (Total risk): ${ (y * amt).sum() }")
print(f"Ngưỡng điểm tối ưu (Optimal Threshold): {opt_threshold}")

o = np.argsort(-s, kind="stable")
ss, ys, am = s[o], y[o], amt[o]
total = float((y * amt).sum())
costs = total - EFFECTIVENESS * np.cumsum(ys * am) + REVIEW_COST * np.arange(1, len(s) + 1)

print("BẢNG TÍNH CHI PHÍ THEO TỪNG NGHƯỠNG")
print("Top K | Score | Is Fraud? | Amount | Cum Fraud Saved | Review Cost | Total Cost")
print("-" * 75)
cum_saved = 0
for i in range(len(s)):
    if ys[i] == 1:
        cum_saved += am[i] * EFFECTIVENESS
    rev_cost = (i + 1) * REVIEW_COST
    c = total - cum_saved + rev_cost
    print(f"  {i+1}   | {ss[i]:.2f}  |    {ys[i]}      | ${am[i]:<5} | ${cum_saved:<14.1f} | ${rev_cost:<11.1f} | ${c:.1f}")

Tổng gian lận ban đầu (Total risk): $700
Ngưỡng điểm tối ưu (Optimal Threshold): 0.6
BẢNG TÍNH CHI PHÍ THEO TỪNG NGHƯỠNG
Top K | Score | Is Fraud? | Amount | Cum Fraud Saved | Review Cost | Total Cost
---------------------------------------------------------------------------
  1   | 0.95  |    1      | $500   | $500.0          | $5.0         | $205.0
  2   | 0.80  |    0      | $50    | $500.0          | $10.0        | $210.0
  3   | 0.60  |    1      | $200   | $700.0          | $15.0        | $15.0
  4   | 0.40  |    0      | $1000  | $700.0          | $20.0        | $20.0
  5   | 0.10  |    0      | $20    | $700.0          | $25.0        | $25.0


In [10]:
def f1_opt_threshold(y, s):
    P, R, T = precision_recall_curve(y, s)
    f1 = 2 * P[:-1] * R[:-1] / np.clip(P[:-1] + R[:-1], 1e-12, None)
    return T[np.argmax(f1)]

y = np.array([1,    0,    1,    0,    1,    0,    0,    1])
s = np.array([0.92, 0.85, 0.78, 0.60, 0.55, 0.40, 0.20, 0.15])
opt_t = f1_opt_threshold(y, s)
print(f"Ngưỡng F1 tối ưu tìm được: {opt_t:.4f}\n")

P, R, T = precision_recall_curve(y, s)
f1 = 2 * P[:-1] * R[:-1] / np.clip(P[:-1] + R[:-1], 1e-12, None)

print("BẢNG TÍNH F1-SCORE THEO TỪNG NGƯỠNG")
print("Threshold | Precision | Recall   | F1-Score")
print("-" * 43)
for idx in range(len(T)):
    marker = " <-- MAX F1" if T[idx] == opt_t else ""
    print(f"  {T[idx]:.2f}    |   {P[idx]:.4f}  |  {R[idx]:.4f}  |  {f1[idx]:.4f}{marker}")

Ngưỡng F1 tối ưu tìm được: 0.1500

BẢNG TÍNH F1-SCORE THEO TỪNG NGƯỠNG
Threshold | Precision | Recall   | F1-Score
-------------------------------------------
  0.15    |   0.5000  |  1.0000  |  0.6667 <-- MAX F1
  0.20    |   0.4286  |  0.7500  |  0.5455
  0.40    |   0.5000  |  0.7500  |  0.6000
  0.55    |   0.6000  |  0.7500  |  0.6667
  0.60    |   0.5000  |  0.5000  |  0.5000
  0.78    |   0.6667  |  0.5000  |  0.5714
  0.85    |   0.5000  |  0.2500  |  0.3333
  0.92    |   1.0000  |  0.2500  |  0.4000


In [11]:
y_true = np.array([0,    1,    0,    1])
y_scores = np.array([0.1,  0.4,  0.35, 0.8])

P, R, T = precision_recall_curve(y_true, y_scores)

print(f"Số lượng Thresholds (T): {len(T)}")
print(f"Số lượng Precision (P) : {len(P)}")
print(f"Số lượng Recall (R)    : {len(R)}")
print("-" * 55)

print("BẢNG TỎA ĐỘ (P, R) VÀ NGƯỠNG TƯƠNG ỨNG:")
for i in range(len(P)):
    if i < len(T):
        t_str = f"T[{i}] = {T[i]:.2f}"
        note = f"Xét giao dịch có score >= {T[i]:.2f}"
    else:
        t_str = "T[end] = Inf"
        note = "Ngưỡng > max(score) -> Không chọn mẫu nào (Default end-point)"
    print(f"Index {i}:  Precision = {P[i]:.2f} | Recall = {R[i]:.2f} | {t_str:<14} ({note})")

Số lượng Thresholds (T): 4
Số lượng Precision (P) : 5
Số lượng Recall (R)    : 5
-------------------------------------------------------
BẢNG TỎA ĐỘ (P, R) VÀ NGƯỠNG TƯƠNG ỨNG:
Index 0:  Precision = 0.50 | Recall = 1.00 | T[0] = 0.10    (Xét giao dịch có score >= 0.10)
Index 1:  Precision = 0.67 | Recall = 1.00 | T[1] = 0.35    (Xét giao dịch có score >= 0.35)
Index 2:  Precision = 1.00 | Recall = 1.00 | T[2] = 0.40    (Xét giao dịch có score >= 0.40)
Index 3:  Precision = 1.00 | Recall = 0.50 | T[3] = 0.80    (Xét giao dịch có score >= 0.80)
Index 4:  Precision = 1.00 | Recall = 0.00 | T[end] = Inf   (Ngưỡng > max(score) -> Không chọn mẫu nào (Default end-point))


In [12]:
def money(y, amt, flag):
    flag = np.asarray(flag, bool)
    loss = float(amt[y == 1].sum())
    prevented = EFFECTIVENESS * float(amt[flag & (y == 1)].sum())
    review = REVIEW_COST * int(flag.sum())
    return dict(
        loss_no_model=loss,
        prevented=prevented,
        review_cost=review,
        net_saved=prevented - review,
        saved_pct=(prevented - review) / loss if loss > 0 else 0.0
    )

y   = np.array([1,    0,    1,    0,    0])
s   = np.array([0.95, 0.80, 0.60, 0.40, 0.10])
amt = np.array([500,  50,   200,  1000, 20])

threshold = 0.70
flag = s >= threshold
report = money(y, amt, flag)

print(f"BÁO CÁO TÀI CHÍNH KHI DÙNG NGƯỠNG {threshold}")
for key, value in report.items():
    if key == "saved_pct":
        print(f"{key:<15}: {value * 100:.2f}%")
    else:
        print(f"{key:<15}: ${value:,.2f}")

BÁO CÁO TÀI CHÍNH KHI DÙNG NGƯỠNG 0.7
loss_no_model  : $700.00
prevented      : $500.00
review_cost    : $10.00
net_saved      : $490.00
saved_pct      : 70.00%


In [16]:
def make_data(num, cat):
    pre = ColumnTransformer([("num", StandardScaler(), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cat)]) if cat else \
          ColumnTransformer([("num", StandardScaler(), num)])
    Xtr = pre.fit(tr[num + cat]).transform(tr[num + cat]).astype(np.float32)        
    Xva = pre.transform(va[num + cat]).astype(np.float32)
    return pre, Xtr, Xva

tr = pd.DataFrame({
    "amount": [100.0, 500.0, 50.0, 200.0],
    "category": ["grocery", "tech", "grocery", "fashion"]
})

va = pd.DataFrame({
    "amount": [300.0, 150.0],
    "category": ["tech", "unknown_cat"] # "unknown_cat" là danh mục mới chưa có trong Train
})

num_cols = ["amount"]
cat_cols = ["category"]

pre_processor, Xtr, Xva = make_data(num_cols, cat_cols)

print(pre_processor)
print("MA TRẬN DỮ LIỆU TẬP TRAIN (Xtr)")
print(Xtr)
print(" MA TRẬN DỮ LIỆU TẬP VALIDATION (Xva)")
print(Xva)

ColumnTransformer(transformers=[('num', StandardScaler(), ['amount']),
                                ('cat',
                                 OneHotEncoder(dtype=<class 'numpy.float32'>,
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['category'])])
MA TRẬN DỮ LIỆU TẬP TRAIN (Xtr)
[[-0.6445034   0.          1.          0.        ]
 [ 1.6470642   0.          0.          1.        ]
 [-0.93094933  0.          1.          0.        ]
 [-0.07161149  1.          0.          0.        ]]
 MA TRẬN DỮ LIỆU TẬP VALIDATION (Xva)
[[ 0.5012804   0.          0.          1.        ]
 [-0.35805744  0.          0.          0.        ]]


In [18]:
FACTORY = {
    "RF": RandomForestClassifier,
    "SVM_Linear": SVC
}

SVM_IDX = np.array([0, 1, 2])
ytr = np.array([0, 1, 0, 1, 0]) 

def score(m, X): 
    return m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)

def fit_score(name, params, Xtr_, Xva_):
    idx = SVM_IDX if name.startswith("SVM") else slice(None)
    m = FACTORY[name](**params).fit(Xtr_[idx], ytr[idx])
    return m, score(m, Xva_)

Xtr_dummy = np.random.randn(5, 4) 
Xva_dummy = np.random.randn(2, 4)

# Case 1: Chạy Random Forest (Dùng 100% data Train = 5 mẫu, xuất ra xác suất predict_proba)
rf_model, s_val_rf = fit_score("RF", {"n_estimators": 10}, Xtr_dummy, Xva_dummy)
print("RF Validation Scores (Probabilities):", s_val_rf)

# Case 2: Chạy SVM (Chỉ dùng SVM_IDX = 3 mẫu, xuất ra decision_function)
svm_model, s_val_svm = fit_score("SVM_Linear", {"kernel": "linear"}, Xtr_dummy, Xva_dummy)
print("SVM Validation Scores (Distance Scores):", s_val_svm)

RF Validation Scores (Probabilities): [0.5 0.3]
SVM Validation Scores (Distance Scores): [ 0.83511833 -0.11557507]


In [20]:

np.random.seed(42)
N_samples = 1000

# Giả sử LightGBM tiết kiệm trung bình $120,000 (độ lệch $5,000)
# Giả sử SVM tiết kiệm trung bình $115,000 (độ lệch $6,000)
B = {
    "net_saved": {
        "LightGBM": np.random.normal(120000, 5000, N_samples),
        "SVM": np.random.normal(115000, 6000, N_samples)
    }
}

def paired(a, b, k):
    d = B[k][a] - B[k][b]
    return {
        "mean_diff": d.mean(),
        "CI_lo": np.percentile(d, 2.5),
        "CI_hi": np.percentile(d, 97.5),
        "P(a>b)": (d > 0).mean()
    }

res = paired("LightGBM", "SVM", "net_saved")

print("KẾT QUẢ SO SÁNH BOOTSTRAP (LightGBM vs SVM)")
print(f"Tiền tiết kiệm chênh lệch trung bình : ${res['mean_diff']:,.2f}")
print(f"Khoảng tin cậy 95% (95% CI)          : [${res['CI_lo']:,.2f}  đến  ${res['CI_hi']:,.2f}]")
print(f"Xác suất LightGBM tốt hơn SVM P(A>B) : {res['P(a>b)'] * 100:.1f}%")

KẾT QUẢ SO SÁNH BOOTSTRAP (LightGBM vs SVM)
Tiền tiết kiệm chênh lệch trung bình : $4,671.64
Khoảng tin cậy 95% (95% CI)          : [$-10,680.68  đến  $19,866.37]
Xác suất LightGBM tốt hơn SVM P(A>B) : 70.8%
